# Notebook Configuration

In [1]:
# General Configuration Variables
DATASET_RESULT_DIR = "synthetic-dataset/data"
TOPOLOGIES_RESULT_DIR = "synthetic-dataset/synthetic-topologies"
PROBLEM_INSTANCES_DIR = "problem-instances"
# DEVICES_DATASET_PATH = "eua-dataset/edge-servers/site-optus-melbCBD.csv"
DEVICES_DATASET_PATH = "eua-dataset/edge-servers/site.csv"
VENDORS_TO_CONSIDER = ["Telstra", "Optus", "Vodafone", "Telecom", "Macquarie"]
UNLIMITED_VALUE = 100000000

# Import the local package (reload if already imported)
import sys
if 'pricing_driven_service_allocation' in sys.modules:
    import importlib
    import pricing_driven_service_allocation
    importlib.reload(pricing_driven_service_allocation)
    # Also reload submodules
    if hasattr(pricing_driven_service_allocation, 'generators'):
        importlib.reload(pricing_driven_service_allocation.generators)
else:
    import pricing_driven_service_allocation as pdsa

import pricing_driven_service_allocation as pdsa

# Synthetic Dataset Generation

In [2]:
import os

if not os.path.exists(DATASET_RESULT_DIR):
  os.makedirs(DATASET_RESULT_DIR)


if not os.path.exists(TOPOLOGIES_RESULT_DIR):
  os.makedirs(TOPOLOGIES_RESULT_DIR)
  
if not os.path.exists(PROBLEM_INSTANCES_DIR):
  os.makedirs(PROBLEM_INSTANCES_DIR)

In [3]:
# Install required dependencies
!pip install shapely -q

In [4]:
import pandas as pd

# Load devices using the package function
devices_df = pdsa.dataset.load_devices_dataframe(DEVICES_DATASET_PATH)

print("Dataset size:", len(devices_df))
devices_df.head()

Dataset size: 95562


,name,latitude,longitude,elevation
device_id,,,,
1000,Fort Hill Wharf DARWIN,-12.471947,130.845073,NaN
10000,Cnr Castlereagh & Lethbri PENRITH,-33.756158,150.698182,NaN
10000002,Optus 50m Lattice Tower 71 Eastward Road Utakarra,-28.777660,114.634260,NaN
10000003,6 Knuckey Street Darwin,-12.464597,130.840708,NaN
10000004,Cape Wickham Links Clubhouse KING ISLAND,-39.596400,143.933900,NaN


In [5]:
# Filter devices by vendor names using the package function
devices_df = pdsa.dataset.filter_devices_by_vendors(devices_df, VENDORS_TO_CONSIDER)

print("Total devices after filtering:", len(devices_df))
devices_df

Total devices after filtering: 18822


,latitude,longitude,elevation,provider
device_id,,,,
10000002,-28.777660,114.634260,NaN,OPTUS
100001,-38.248652,144.605442,23.0,TELSTRA
10000114,-31.901910,152.533540,NaN,OPTUS
100002,-37.728550,145.222007,116.0,OPTUS
10000215,-32.981570,121.644400,NaN,TELSTRA
...,...,...,...,...
9954,-34.819950,147.902049,714.0,TELSTRA
9958,-34.971752,147.998115,709.0,TELSTRA
9967,-36.130494,144.750901,100.0,TELSTRA


In [6]:
devices_df.to_csv(os.path.join(DATASET_RESULT_DIR, "devices.csv"))

In [7]:
# Custom configuration for resource assignment
custom_config = {
    'global': {
        'group_percentages': {1: 83, 2: 15, 3: 2},
        'group_ranges': {1: (0, 12.5), 2: (12.5, 50), 3: (50, 100)}
    },
    'attributes': {
        'available_RAM': {
            'min': 1,
            'max': 128,
            'default_price': 0.5,
            'price_by_provider_group': {
                'OPTUS': {1: 1, 2: 0.15, 3: 0.005},
                'TELSTRA': {1: 1.3, 2: 0.12, 3: 0.003},
                'VODAFONE': {1: 1.5, 2: 0.18, 3: 0.001},
                'MACQUARIE': {1: 0.2, 2: 0.17, 3: 0.008},
                'TELECOM': {1: 0.7, 2: 0.14, 3: 0.004},
            },
            'local_distribution': {
                1: [(70, 0, 25), (30, 25, 100)]
            }
        },
        'available_Storage': {
            'min': 1,
            'max': 100000,
            'default_price': 0.2,
            'price_by_provider_group': {
                'OPTUS': {1: 0.90, 2: 0.25, 3: 0.08},
                'TELSTRA': {1: 1.10, 2: 0.30, 3: 0.10},
                'VODAFONE': {1: 1.25, 2: 0.35, 3: 0.05},
                'MACQUARIE': {1: 0.50, 2: 0.20, 3: 0.09},
                'TELECOM': {1: 0.85, 2: 0.28, 3: 0.07},
            },
            'local_distribution': {
                1: [(70, 0, 0.016), (30, 0.512, 16.384)]
            }
        },
        'available_vCPU': {
            'min': 1,
            'max': 64,
            'default_price': 15
        },
        'available_GPU': {
            'min': 0,
            'max': 8,
            'default_price': 120,
            'price_by_provider_group': {
                'TELSTRA': {2: 130, 3: 150},
                'OPTUS': {3: 140}
            },
            'local_distribution': {
                1: [(85, 0, 25), (15, 25, 50)],
                2: [(60, 0, 50), (40, 50, 100)],
                3: [(30, 0, 50), (70, 50, 100)]
            }
        },
        'available_TPU': {
            'min': 0,
            'max': 4,
            'default_price': 200,
            'price_by_provider_group': {
                'TELSTRA': {3: 240},
                'OPTUS': {2: 210}
            },
            'local_distribution': {
                1: [(90, 0, 25), (10, 25, 50)],
                2: [(70, 0, 50), (30, 50, 100)],
                3: [(40, 0, 50), (60, 50, 100)]
            }
        }
    },
    'devices_types_by_group': {
        1: {'SENSOR': 25, 'CAMERA': 50, 'MOBILE': 25},
        2: {'LAPTOP': 25, 'COMPUTER': 50, 'MOBILE': 25},
        3: {'DATA_CENTER': 75, 'SERVER': 25}
    }
}

# Apply resource assignment using the package function
devices_df = pdsa.dataset.assign_device_resources(devices_df, custom_config)

print(f"\nDevices per group:")
print(devices_df['global_group'].value_counts().sort_index())
print(f"\nResource statistics:")
print(devices_df[['available_RAM', 'available_Storage', 'available_vCPU', 'available_GPU', 'available_TPU']].describe())
print(f"\nAccelerator availability (counts):")
print(f"available_GPU: {devices_df['available_GPU'].sum()}")
print(f"available_TPU: {devices_df['available_TPU'].sum()}")

# Show pricing overrides applied on a sample
price_cols = [c for c in devices_df.columns if c.startswith('unit_price_')]
print("\nSample unit prices:")
print(devices_df[['provider', 'global_group'] + price_cols].head(10))

devices_df.head(10)


Devices per group:
global_group
1    15622
2     2823
3      377
Name: count, dtype: int64

Resource statistics:
       available_RAM  available_Storage  available_vCPU  available_GPU  \
count   18822.000000       18822.000000    18822.000000   18822.000000   
mean       12.014079        6526.187600        7.645362       0.398310   
std        18.752612       15597.593057        8.809425       1.087062   
min         1.000000           1.000000        1.000000       0.000000   
25%         2.000000           1.000000        3.000000       0.000000   
50%         4.000000           2.000000        5.000000       0.000000   
75%        13.000000        1432.750000        8.000000       0.000000   
max       127.000000       99884.000000       63.000000       7.000000   

       available_TPU  
count   18822.000000  
mean        0.132398  
std         0.450520  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         0.000000  
max         3.000000  

Accelerator

,latitude,longitude,elevation,provider,global_group,device_type,available_RAM,unit_price_available_RAM,available_Storage,unit_price_available_Storage,available_vCPU,unit_price_available_vCPU,available_GPU,unit_price_available_GPU,available_TPU,unit_price_available_TPU
device_id,,,,,,,,,,,,,,,,
10000002,-28.777660,114.634260,NaN,OPTUS,1,CAMERA,1,1.000,2,0.9,3,15.0,0,120.0,0,200.0
100001,-38.248652,144.605442,23.0,TELSTRA,1,MOBILE,1,1.300,1,1.1,7,15.0,0,120.0,0,200.0
10000114,-31.901910,152.533540,NaN,OPTUS,1,MOBILE,2,1.000,2,0.9,6,15.0,0,120.0,0,200.0
100002,-37.728550,145.222007,116.0,OPTUS,1,CAMERA,4,1.000,1,0.9,6,15.0,0,120.0,0,200.0
10000215,-32.981570,121.644400,NaN,TELSTRA,3,SERVER,4,0.003,1,0.1,6,15.0,0,150.0,0,240.0
10000216,-35.021970,149.872360,NaN,TELSTRA,1,CAMERA,1,1.300,1,1.1,2,15.0,0,120.0,0,200.0
10000264,-36.706940,144.317030,NaN,TELSTRA,1,MOBILE,4,1.300,2,1.1,5,15.0,0,120.0,0,200.0
10000265,-24.182717,151.236237,NaN,TELSTRA,1,MOBILE,1,1.300,1,1.1,6,15.0,0,120.0,0,200.0
10000272,-26.204260,152.442950,NaN,TELSTRA,1,MOBILE,4,1.300,1,1.1,4,15.0,0,120.0,0,200.0


In [8]:
devices_df.to_csv(os.path.join(DATASET_RESULT_DIR, "devices.csv"))

# Topologies Generator

In [9]:
# Example: Generate topology for Melbourne CBD using the package function
# Parameters:
# - Center: Melbourne CBD (-37.8136, 144.9631)
# - Radius: 15000 meters (15 km)
# - Max providers: 3
# - Allowed groups: [1, 2, 3] (low, medium, and high capacity devices)
# - Number of devices: 1500

topology_devices, topology_id = pdsa.generators.topology(
    lat=-37.8136,
    long=144.9631,
    rad=15000,
    devices_df=devices_df,
    topologies_result_dir=TOPOLOGIES_RESULT_DIR,
    number_of_providers=3,
    allowed_groups=[1, 2, 3],
    number_of_devices=1500
)

print(f"\nTopology ID: {topology_id}")
print(f"\nTopology devices summary:")
print(topology_devices[['provider', 'global_group', 'device_type', 'available_RAM', 'available_Storage', 'available_vCPU', 'available_GPU', 'available_TPU']].describe())

Topology 85c25b0a-3558-4bb0-a5ec-ae6b79a1763f generated successfully!
  - Devices selected: 1098
  - Providers: 3 (TELSTRA, OPTUS, VODAFONE)
  - Groups: [1, 2, 3]
  - Files saved in: synthetic-dataset/synthetic-topologies/85c25b0a-3558-4bb0-a5ec-ae6b79a1763f

Topology ID: 85c25b0a-3558-4bb0-a5ec-ae6b79a1763f

Topology devices summary:
       global_group  available_RAM  available_Storage  available_vCPU  \
count   1098.000000    1098.000000        1098.000000     1098.000000   
mean       1.188525       9.547359        4814.988160        6.750455   
std        0.443710      15.759821       12781.608878        6.979957   
min        1.000000       1.000000           1.000000        1.000000   
25%        1.000000       2.000000           1.000000        3.000000   
50%        1.000000       3.000000           2.000000        5.000000   
75%        1.000000      10.000000         924.750000        8.000000   
max        3.000000     124.000000       93282.000000       56.000000   

     

# Pricing Generator

In [10]:
# Example 1: Generate pricing without provider compatibility using the package function
print("=" * 80)
print("Example 1: Pricing with provider isolation")
print("=" * 80)
pricing_path_1 = pdsa.generators.pricing_from_topology(
    topology_id=topology_id,
    topologies_result_dir=TOPOLOGIES_RESULT_DIR
)

# Example 2: Generate pricing with compatible provider groups
# print("\n" + "=" * 80)
# print("Example 2: Pricing with provider groups (OPTUS+TELSTRA compatible)")
# print("=" * 80)
# compatible_groups = [["OPTUS", "TELSTRA"], ["VODAFONE"]]
# pricing_path_2 = pdsa.generators.pricing(
#     topology_id=topology_id,
#     topologies_result_dir=TOPOLOGIES_RESULT_DIR,
#     compatible_provider_groups=compatible_groups
# )

Example 1: Pricing with provider isolation
Pricing YAML generated successfully!
  - Topology ID: 85c25b0a-3558-4bb0-a5ec-ae6b79a1763f
  - Devices: 1098
  - Providers: ['TELSTRA', 'OPTUS', 'VODAFONE']
  - File saved: synthetic-dataset/synthetic-topologies/85c25b0a-3558-4bb0-a5ec-ae6b79a1763f/pricing.yml


# Generating iPricing Model in Python

This section provides the required functions to map any given problem instance represented as an iPricing into a Python class.

In [11]:
import os

if not os.path.exists("./iPricing/model"):
    os.makedirs("./iPricing/model")

!protoc --python_out=./iPricing/model ./iPricing/iPricing.proto
!mv ./iPricing/model/iPricing/iPricing_pb2.py ./iPricing/model/iPricing_pb2.py
!rm -rf ./iPricing/model/iPricing

In [12]:
import os
from iPricing.model.iPricing_pb2 import Pricing

# Convert YAML to Protocol Buffer using the package function
pricing_obj = pdsa.utils.yaml_to_pricing_proto(
    os.path.join(TOPOLOGIES_RESULT_DIR, topology_id, "pricing.yml"), 
    Pricing
)

print(f"Saved {pricing_obj.saasName} pricing as object")

Saved 85c25b0a-3558-4bb0-a5ec-ae6b79a1763f pricing as object


# Processing of User Requests

This section defines the function that integrates user service requests with the pricing parameters of a given topology *T*. By consolidating these data sources, the function generates a comprehensive problem instance that serves as the input for the optimization engine, enabling the determination of the optimal system configuration.

In [13]:
import json

# Custom request configuration
custom_request = {
    'currency': 'USD',
    'users_location': [
        # This limits to melbourne area
        (144.8588538568788, -37.84750988098617),
        (144.8588538568788, -37.857590074634246),
        (144.87928006127504, -37.857590074634246),
        (144.87928006127504, -37.84750988098617),
        (144.8588538568788, -37.84750988098617)
    ],
    'providers_to_consider': ['telstra'],
    'budget': 1000,
    'max_devices': 2,
    'device_types': ['CAMERA', 'SENSOR', 'DATA_CENTER'],
    'resources': {
        'available_ram': 5,
        'available_storage': 500,
        'available_vcpu': 4,
        'available_gpu': 1,
        'available_tpu': 0,
    },
    'max_distance': 10000,  # in meters
}

# Generate problem instance using the package function
problem_instance_pricing, filter_criteria = pdsa.generators.problem_instance(
    instance_pricing=pricing_obj,
    request=custom_request,
    topologies_result_dir=TOPOLOGIES_RESULT_DIR,
    unlimited_value=UNLIMITED_VALUE
)

print("\nGenerated problem instance pricing and filter criteria:")
print(f"- Original number of add-ons: {len(pricing_obj.addOns)}")
print(f"- Number of add-ons: {len(problem_instance_pricing.addOns)}")
print(f"- Filter criteria: {json.dumps(filter_criteria, indent=2)}")


Generated problem instance pricing and filter criteria:
- Original number of add-ons: 1098
- Number of add-ons: 673
- Filter criteria: {
  "maxPrice": 1000,
  "maxSubscriptionSize": 2,
  "usageLimits": {
    "available_ram": 5,
    "available_storage": 500,
    "available_vcpu": 4,
    "available_gpu": 1,
    "available_tpu": 0,
    "distance": 99990000
  },
  "features": [
    "CAMERA",
    "SENSOR",
    "DATA_CENTER"
  ]
}


In [14]:
# Save the generated pricing using the package function
pdsa.utils.pricing_proto_to_yaml(
    problem_instance_pricing, 
    os.path.join(TOPOLOGIES_RESULT_DIR, topology_id, "problem_instance_pricing.yml")
)

Generated pricing saved to: synthetic-dataset/synthetic-topologies/85c25b0a-3558-4bb0-a5ec-ae6b79a1763f/problem_instance_pricing.yml


# Problem Resolution

In [15]:
# ----- FIRST ITERATION OF THE PROBLEM -----

# Location of users
  # Either as coordinates (a point)
    # Latitude
    # Longitude
  # or in zones (a geographic area)
# Providers to consider
# Maximum budget (mandatory)
# Maximum number of devices (if not set, max = total available devices)
# Minimum resources (if not set, all resources set to 0)
# Maximum distance to users (if not set, max = infinite; declared in meters)

# ----- BACKLOG OF THE PROBLEM -----

# Maximum number of providers
# Maximum distance between the different nodes of S
# Set of services to host
  # Requirements of each service
  # Distance constraints between services